# PathSnap dataset EDA (Member 1A)

Run the audit and cleaning scripts first. This notebook reads the local `cleaned_dataset/` and the source `trip_inventory.csv`. It summarizes the supplied training and student-testing partitions separately. It does not use evaluator-testing data.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import folium

DATA_ROOT = Path('cleaned_dataset')
INVENTORY_PATH = Path('trip_inventory.csv')
inventory = pd.read_csv(INVENTORY_PATH, dtype={'trip_id': str})
inventory.head()

## Trip counts, GPS observations, nodes, and arcs

In [ ]:
summary = inventory.groupby('partition').agg(
    trips=('trip_id', 'count'),
    gps_observations=('track_rows', 'sum'),
    nodes=('nodes', 'sum'),
    arcs=('arcs', 'sum'),
    route_entries=('route_rows', 'sum'),
).reset_index()
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for partition, group in inventory.groupby('partition'):
    axes[0].plot(group['trip_id'], group['track_rows'], marker='.', linestyle='none', label=partition)
axes[0].set(title='GPS observations per trip', xlabel='Trip ID', ylabel='Observations')
axes[0].tick_params(axis='x', rotation=90)
axes[0].legend()
for partition, group in inventory.groupby('partition'):
    axes[1].scatter(group['nodes'], group['arcs'], label=partition, alpha=.8)
axes[1].set(title='Road-network size per trip', xlabel='Node rows', ylabel='Arc rows')
axes[1].legend()
plt.tight_layout()
plt.show()

## Timestamp intervals and GPS gaps

In [ ]:
interval_records = []
for row in inventory.itertuples(index=False):
    track_path = DATA_ROOT / row.partition / row.trip_id / f'{row.trip_id}.track'
    track = pd.read_csv(track_path, sep=r'\s+', header=None, names=['lon', 'lat', 'timestamp'])
    intervals = track['timestamp'].diff().dropna()
    interval_records.extend({'partition': row.partition, 'trip_id': row.trip_id, 'gap_s': value} for value in intervals)
intervals = pd.DataFrame(interval_records)
display(intervals.groupby('partition')['gap_s'].describe())
fig, ax = plt.subplots(figsize=(9, 5))
for partition, group in intervals.groupby('partition'):
    ax.hist(group['gap_s'].clip(upper=30), bins=60, alpha=.55, label=partition)
ax.set(title='Timestamp intervals (values above 30s shown in final bin)', xlabel='Seconds between adjacent observations', ylabel='Count')
ax.legend()
plt.tight_layout()
plt.show()

## Example GPS trajectory plot

In [ ]:
EXAMPLE_PARTITION = 'train'
EXAMPLE_TRIP = '00000001'
trip_dir = DATA_ROOT / EXAMPLE_PARTITION / EXAMPLE_TRIP
track = pd.read_csv(trip_dir / f'{EXAMPLE_TRIP}.track', sep=r'\s+', header=None, names=['lon', 'lat', 'timestamp'])
plt.figure(figsize=(7, 7))
plt.plot(track['lon'], track['lat'], '-', linewidth=.8, alpha=.75)
plt.scatter(track['lon'], track['lat'], s=5)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title(f'GPS trajectory: {EXAMPLE_PARTITION}/{EXAMPLE_TRIP}')
plt.axis('equal')
plt.tight_layout()
plt.show()

## Folium map: GPS observations, nearby road network, and supplied reference route

In [ ]:
nodes = pd.read_csv(trip_dir / f'{EXAMPLE_TRIP}.nodes', sep=r'\s+', header=None, names=['lon', 'lat'])
arcs = pd.read_csv(trip_dir / f'{EXAMPLE_TRIP}.arcs', sep=r'\s+', header=None, names=['source', 'target']).astype(int)
route = pd.read_csv(trip_dir / f'{EXAMPLE_TRIP}.route', sep=r'\s+', header=None, names=['arc_id'])['arc_id'].astype(int).tolist()
pad = 0.003
west, east = track['lon'].min() - pad, track['lon'].max() + pad
south, north = track['lat'].min() - pad, track['lat'].max() + pad
center = [track['lat'].mean(), track['lon'].mean()]
map_view = folium.Map(location=center, zoom_start=14, control_scale=True)
gps_layer = folium.FeatureGroup(name='GPS observations', show=True)
gps_locations = track[['lat', 'lon']].values.tolist()
folium.PolyLine(gps_locations, color='blue', weight=3, opacity=.75, tooltip='GPS trajectory').add_to(gps_layer)
for lat, lon in gps_locations:
    folium.CircleMarker([lat, lon], radius=2, color='blue', fill=True, fill_opacity=.7, weight=1).add_to(gps_layer)
gps_layer.add_to(map_view)

roads_layer = folium.FeatureGroup(name='Road network near trajectory', show=True)
road_count = 0
for arc in arcs.itertuples(index=False):
    a, b = nodes.iloc[arc.source], nodes.iloc[arc.target]
    if max(a.lon, b.lon) < west or min(a.lon, b.lon) > east or max(a.lat, b.lat) < south or min(a.lat, b.lat) > north:
        continue
    folium.PolyLine([[a.lat, a.lon], [b.lat, b.lon]], color='gray', weight=1, opacity=.45).add_to(roads_layer)
    road_count += 1
roads_layer.add_to(map_view)

route_layer = folium.FeatureGroup(name='Supplied reference route', show=True)
for arc_id in route:
    source, target = arcs.iloc[arc_id]
    a, b = nodes.iloc[source], nodes.iloc[target]
    folium.PolyLine([[a.lat, a.lon], [b.lat, b.lon]], color='green', weight=4, opacity=.9).add_to(route_layer)
route_layer.add_to(map_view)
folium.LayerControl(collapsed=False).add_to(map_view)
print(f'Displayed {road_count:,} nearby road arcs; reference arcs are drawn individually to avoid inventing links at route discontinuities.')
map_view

## Interpretation notes

The map shows the supplied reference arc sequence as individual directed arcs; it does not force disconnected consecutive arcs into an artificial line. Long timestamp gaps are preserved and should be handled explicitly downstream. The student-testing partition is included only for development-stage summaries and comparisons.